# Multilingual Verifier Fine-Tuning
This notebook fine-tunes a multilingual HuggingFace encoder model for verifier scoring using multilingual trace-level data.

In [18]:
import importlib

try:
    drive = importlib.import_module("google.colab").drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print("Google Colab is not available here; skipping drive mount.")

Google Colab is not available here; skipping drive mount.


In [19]:
%pip install -q transformers evaluate tomli sentencepiece accelerate

Note: you may need to restart the kernel to use updated packages.


In [20]:
import os
import re
import json
import time
import random
import datetime
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split


import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
)

from sklearn.metrics import accuracy_score
import evaluate


In [21]:
torch.set_default_dtype(torch.float32)


In [22]:
f1_metric = evaluate.load("f1")

def format_time(elapsed):
    elapsed_rounded = int(round(elapsed))
    return str(datetime.timedelta(seconds=elapsed_rounded))


def remove_label_pattern(text):
    text = re.sub(
        r"(\[?\s*Justification\s*\]?:?\s*)|(\[?\s*Label\s*\]?:\s*(True|False|Conflicting|Unknown|Supports|Refutes))",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()
    return text.replace("\n", " ")


def print_trainable_parameters(model):
    trainable_params, all_param = 0, 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()

    print(
        f"trainable params: {trainable_params} || "
        f"all params: {all_param} || "
        f"trainable%: {100 * trainable_params / all_param:.2f}"
    )


In [23]:
class CustomClassifier(torch.nn.Module):
    def __init__(
        self,
        model_name,
        num_labels=1,
        freeze_base_layer=False,
    ):
        super().__init__()

        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=num_labels,
        )

        if freeze_base_layer:
            for param in self.model.parameters():
                param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        return outputs.logits


In [24]:
class CustomClassifier(torch.nn.Module):
    def __init__(
        self,
        model_name,
        num_labels=1,
        use_lora=False,
        is_base_encoder=True,
        freeze_base_layer=False,
    ):
        super().__init__()

        self.use_lora = use_lora
        self.is_base_encoder = is_base_encoder

        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=num_labels,
        )

        if freeze_base_layer:
            for param in self.model.parameters():
                param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        return outputs.logits

In [25]:
class TrainerModule:
    def __init__(
        self,
        model,
        train_loader,
        val_loader,
        epochs,
        lr,
        patience,
        output_dir,
        tokenizer,
        grad_accum_steps=1,
        use_amp=False,
    ):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)

        # Try to enable gradient checkpointing on the underlying base model (if supported)
        try:
            if hasattr(self.model, "gradient_checkpointing_enable"):
                self.model.gradient_checkpointing_enable()
            elif hasattr(self.model, "base_model") and hasattr(self.model.base_model, "gradient_checkpointing_enable"):
                self.model.base_model.gradient_checkpointing_enable()
        except Exception:
            pass

        self.train_loader = train_loader
        self.val_loader = val_loader
        self.epochs = epochs
        self.patience = patience
        self.output_dir = output_dir
        self.tokenizer = tokenizer

        self.grad_accum_steps = grad_accum_steps
        self.use_amp = use_amp and (self.device.type == "cuda" and torch.cuda.is_available())
        self.scaler = torch.amp.GradScaler("cuda") if self.use_amp else None

        # Only optimize parameters that require gradients (e.g., LoRA adapters)
        self.optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, eps=1e-8)
        self.loss_fn = torch.nn.BCEWithLogitsLoss()

        # Estimate total steps taking gradient accumulation into account
        iters_per_epoch = max(1, len(train_loader))
        total_steps = (iters_per_epoch // max(1, self.grad_accum_steps) + (1 if iters_per_epoch % max(1, self.grad_accum_steps) else 0)) * self.epochs
        warmup_steps = int(0.05 * total_steps) if total_steps > 0 else 0

        self.scheduler = get_cosine_schedule_with_warmup(
            self.optimizer,
            warmup_steps,
            total_steps,
        )

        os.makedirs(output_dir, exist_ok=True)

    def train(self):
        best_val_f1 = -1.0
        epochs_no_improve = 0

        for epoch in range(self.epochs):
            print(f"\nEpoch {epoch+1}/{self.epochs}")
            self.model.train()

            total_loss, total_acc = 0, 0
            self.optimizer.zero_grad()

            for step, batch in enumerate(tqdm(self.train_loader)):
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)

                # Mixed precision context when available
                with torch.amp.autocast(device_type=self.device.type, enabled=self.use_amp):
                    logits = self.model(input_ids, attention_mask)
                    loss = self.loss_fn(logits.squeeze(1), labels)
                    loss = loss / max(1, self.grad_accum_steps)

                if self.use_amp:
                    self.scaler.scale(loss).backward()
                else:
                    loss.backward()

                # Step optimizer when we've accumulated enough gradients
                if (step + 1) % max(1, self.grad_accum_steps) == 0 or (step + 1) == len(self.train_loader):
                    if self.use_amp:
                        self.scaler.unscale_(self.optimizer)
                        torch.nn.utils.clip_grad_norm_(filter(lambda p: p.requires_grad, self.model.parameters()), max_norm=1.0)
                        self.scaler.step(self.optimizer)
                        self.scaler.update()
                    else:
                        torch.nn.utils.clip_grad_norm_(filter(lambda p: p.requires_grad, self.model.parameters()), max_norm=1.0)
                        self.optimizer.step()

                    if hasattr(self, "scheduler") and self.scheduler is not None:
                        self.scheduler.step()

                    self.optimizer.zero_grad()

                total_loss += loss.item() * max(1, self.grad_accum_steps)

                preds = (torch.sigmoid(logits.squeeze(1)) >= 0.5).cpu().numpy().astype(int)
                total_acc += accuracy_score(labels.cpu().numpy(), preds)

            print(f"Train Loss: {total_loss / len(self.train_loader):.4f}")
            print(f"Train Acc: {total_acc / len(self.train_loader):.4f}")

            val_f1 = self.evaluate(epoch)

            # Early stopping / save best model by val F1
            if val_f1 is not None:
                if val_f1 > best_val_f1:
                    best_val_f1 = val_f1
                    epochs_no_improve = 0
                    ckpt_dir = os.path.join(self.output_dir, "best_checkpoint")
                    os.makedirs(ckpt_dir, exist_ok=True)
                    self.tokenizer.save_pretrained(ckpt_dir)
                    torch.save(
                        self.model.state_dict(),
                        os.path.join(ckpt_dir, "pytorch_model.bin"),
                    )
                    print(f"Saved best model with Val F1: {best_val_f1:.4f} to {ckpt_dir}")
                else:
                    epochs_no_improve += 1

            if epochs_no_improve >= self.patience:
                print("Early stopping triggered.")
                break

    def evaluate(self, epoch):
        self.model.eval()
        total_loss, total_acc = 0, 0
        all_preds, all_labels = [], []

        with torch.no_grad():
            for batch in self.val_loader:
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)

                with torch.amp.autocast(device_type=self.device.type, enabled=self.use_amp):
                    logits = self.model(input_ids, attention_mask)
                    loss = self.loss_fn(logits.squeeze(1), labels)

                total_loss += loss.item()
                preds = (torch.sigmoid(logits.squeeze(1)) >= 0.5).cpu().numpy().astype(int)
                labels_np = labels.cpu().numpy().astype(int)

                total_acc += accuracy_score(labels_np, preds)
                all_preds.extend(preds.tolist())
                all_labels.extend(labels_np.tolist())

        if len(self.val_loader) == 0 or len(all_labels) == 0:
            print("No validation data available to evaluate.")
            return None

        val_f1 = f1_metric.compute(predictions=all_preds, references=all_labels)["f1"]
        print(f"Val Loss: {total_loss / len(self.val_loader):.4f}")
        print(f"Val Acc: {total_acc / len(self.val_loader):.4f}")
        print(f"Val F1: {val_f1:.4f}")

        return val_f1

In [26]:
class VerifierEvaluator:
    def __init__(
        self,
        model_path,
        tokenizer_path,
        base_model,
        use_decomp,
        device="cuda",
    ):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.use_decomp = use_decomp

        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = CustomClassifier(
            base_model,
        )
        self.model.load_state_dict(torch.load(model_path, map_location=self.device))
        self.model.to(self.device)
        self.model.eval()

    def _stringify_evidence(self, evidences):
        if evidences is None:
            return ""
        if isinstance(evidences, str):
            return evidences
        if isinstance(evidences, list):
            return " ".join(str(x).strip() for x in evidences[:3] if str(x).strip())
        return str(evidences)

    def encode_input(self, claim, evidences, verdict, justification, max_length=None):
        evidence_text = self._stringify_evidence(evidences)
        if max_length is None:
            max_length = MAX_LENGTH if "MAX_LENGTH" in globals() else 384

        text = (
            f"Claim: {claim}\n"
            f"Verdict: {verdict}\n"
            f"Evidence: {evidence_text}\n"
            f"Justification: {justification}"
        )

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )

        return (
            encoding["input_ids"].to(self.device),
            encoding["attention_mask"].to(self.device),
        )

    def score(self, claim, evidences, verdict, justification):
        ids, mask = self.encode_input(claim, evidences, verdict, justification)
        with torch.no_grad():
            return float(self.model(ids, mask).item())

In [27]:
# ===== PATH PLACEHOLDERS =====
TRAIN_CSV = "../English/English_train.jsonl"
TRAIN_ARABIC_JSONL = "../Arabic/arabic_train.jsonl"
TRAIN_SPANISH_JSONL = "../spanish/spanish_train.jsonl"
VAL_CLAIMS_JSON = "../English/clef_2026_final_english_test.json"

# Aici se vor salva modelele antrenate (creează un folder local)
OUTPUT_DIR = "./output_models"

# Better scoring choice for multilingual task data on a single GPU laptop
BASE_MODEL = "xlm-roberta-base"

# Laptop (RTX 4060) friendly settings
# Increase if you need less truncation; reduce if you hit OOM.
MAX_LENGTH = 384
BATCH_SIZE = 2
# Accumulate gradients to simulate a larger batch size
GRAD_ACCUM_STEPS = 16
# Use automatic mixed precision on CUDA
USE_AMP = True
EPOCHS = 4
LR = 2e-5

BEST_CHECKPOINT_DIR = f"{OUTPUT_DIR}/best_checkpoint"
PREDICTIONS_JSON = f"{OUTPUT_DIR}/clef_predictions.json"

In [11]:
# Sanity check jsonl inputs before training
required_cols = {"input_text", "Class"}

for name, path in [
    ("English", TRAIN_CSV),
    ("Arabic", TRAIN_ARABIC_JSONL),
    ("Spanish", TRAIN_SPANISH_JSONL),
]:
    df = pd.read_json(path, lines=True)
    missing = required_cols.difference(df.columns)
    if missing:
        raise ValueError(f"{name} jsonl missing columns: {sorted(missing)}")
    print(f"{name} rows: {len(df)}")

English rows: 30659
Arabic rows: 9732
Spanish rows: 8790


In [ ]:
from huggingface_hub import login

hf_token = "token"
if hf_token:
    login(token=hf_token)
else:
    print("HF_TOKEN is not set. Skipping login; this notebook assumes you are already authenticated.")


def load_raw_samples(path, lang):
    with open(path, "r", encoding="utf-8") as f:
        samples = [json.loads(line) for line in f if line.strip()]
    for sample in samples:
        sample["lang"] = lang
    return samples


def build_trace_level_dataframe(raw_samples):
    dataframe = pd.DataFrame(raw_samples).copy()
    required_cols = {"input_text", "Class"}
    missing = required_cols.difference(dataframe.columns)
    if missing:
        raise ValueError(f"Raw samples are missing columns: {sorted(missing)}")
    dataframe = dataframe.dropna(subset=["input_text", "Class"]).reset_index(drop=True)
    dataframe["Class"] = dataframe["Class"].astype(int)
    return dataframe


english_raw = load_raw_samples(TRAIN_CSV, "en")
arabic_raw = load_raw_samples(TRAIN_ARABIC_JSONL, "ar")
spanish_raw = load_raw_samples(TRAIN_SPANISH_JSONL, "es")

all_raw_samples = english_raw + arabic_raw + spanish_raw
raw_labels = [sample["Class"] for sample in all_raw_samples]

train_raw, val_raw = train_test_split(
    all_raw_samples,
    test_size=0.15,
    stratify=raw_labels,
    random_state=42,
)

train_df = build_trace_level_dataframe(train_raw)
val_df = build_trace_level_dataframe(val_raw)

if "TextDataset" not in globals():
    class TextDataset(Dataset):
        def __init__(self, dataframe, tokenizer, max_length):
            self.texts = dataframe["input_text"].tolist()
            self.labels = dataframe["Class"].tolist()
            self.tokenizer = tokenizer
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token
            self.max_length = max_length

        def __len__(self):
            return len(self.texts)

        def __getitem__(self, idx):
            encoding = self.tokenizer(
                self.texts[idx],
                truncation=True,
                padding="max_length",
                max_length=self.max_length,
                return_tensors="pt",
            )
            item = {k: v.squeeze(0) for k, v in encoding.items()}
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
            return item


tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

train_dataset = TextDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = TextDataset(val_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, pin_memory=True)

model = CustomClassifier(
    BASE_MODEL,
    use_lora=False,
    is_base_encoder=True,
)

print_trainable_parameters(model)

trainer = TrainerModule(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    lr=LR,
    patience=2,
    output_dir=OUTPUT_DIR,
    tokenizer=tokenizer,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    use_amp=USE_AMP,
)

trainer.train()

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4127.96it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 278044417 || all params: 278044417 || trainable%: 100.00

Epoch 1/4


100%|██████████| 20902/20902 [21:13<00:00, 16.42it/s]  


Train Loss: 0.5162
Train Acc: 0.7490
Val Loss: 0.4299
Val Acc: 0.8153
Val F1: 0.7041
Saved best model with Val F1: 0.7041 to ./output_models/best_checkpoint

Epoch 2/4


100%|██████████| 20902/20902 [21:24<00:00, 16.28it/s]  


Train Loss: 0.3942
Train Acc: 0.8332
Val Loss: 0.3509
Val Acc: 0.8510
Val F1: 0.7607
Saved best model with Val F1: 0.7607 to ./output_models/best_checkpoint

Epoch 3/4


100%|██████████| 20902/20902 [21:23<00:00, 16.29it/s]  


Train Loss: 0.3218
Train Acc: 0.8613
Val Loss: 0.3070
Val Acc: 0.8673
Val F1: 0.7829
Saved best model with Val F1: 0.7829 to ./output_models/best_checkpoint

Epoch 4/4


100%|██████████| 20902/20902 [21:03<00:00, 16.54it/s]  


Train Loss: 0.2696
Train Acc: 0.8768
Val Loss: 0.3056
Val Acc: 0.8692
Val F1: 0.7882
Saved best model with Val F1: 0.7882 to ./output_models/best_checkpoint


In [14]:
# Write atomically to avoid partial/overwritten outputs if interrupted.
tmp_path = PREDICTIONS_JSON + ".tmp"
with open(tmp_path, "w", encoding="utf-8") as fp:
    json.dump(predictions, fp, indent=4, ensure_ascii=False)
os.replace(tmp_path, PREDICTIONS_JSON)
print(f"Saved {len(predictions)} claim-level predictions to {PREDICTIONS_JSON}")

NameError: name 'predictions' is not defined

In [15]:
with open(PREDICTIONS_JSON, "w", encoding="utf-8") as fp:
    json.dump(predictions, fp, indent=4, ensure_ascii=False)

print(f"Saved {len(predictions)} claim-level predictions to {PREDICTIONS_JSON}")


NameError: name 'predictions' is not defined

In [30]:
# Run predictions on VAL_CLAIMS_JSON (final test) and save atomically
import json
import os
import numpy as np

with open(VAL_CLAIMS_JSON, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"Running predictions on {len(test_data)} samples from {VAL_CLAIMS_JSON}")

evaluator = VerifierEvaluator(
    model_path=f"{BEST_CHECKPOINT_DIR}/pytorch_model.bin",
    tokenizer_path=BEST_CHECKPOINT_DIR,
    base_model=BASE_MODEL,
    use_decomp=False
)

predictions = []

def capitalize_verdict(v):
    v_lower = str(v).lower().strip()
    if v_lower == "false":
        return "False"
    if v_lower == "true":
        return "True"
    if v_lower == "conflicting":
        return "Conflicting"
    return v_lower.capitalize()

for idx, sample in enumerate(test_data):
    verdict_list = []
    verifier_score_list = []
    justification_list = []

    for trace_idx, raw_trace in enumerate(sample.get("Reasoning_traces", [])):
        justification = remove_label_pattern(raw_trace).split("Label:")[0].strip()
        # Some test entries might not have Verdict_list aligned; guard access
        vlist = sample.get("Verdict_list", [])
        verdict = capitalize_verdict(vlist[trace_idx]) if trace_idx < len(vlist) else "Conflicting"

        score = evaluator.score(
            sample.get("claim", ""),
            sample.get("evidences", ""),
            verdict,
            justification,
        )

        verdict_list.append(verdict)
        justification_list.append(justification)
        verifier_score_list.append(score)

    if len(verifier_score_list) == 0:
        # No traces: default to Conflicting
        best_verdict = "Conflicting"
    else:
        ranked_indices = np.argsort(np.array(verifier_score_list))[::-1]
        best_verdict = verdict_list[int(ranked_indices[0])]

    predictions.append({
        "query_id": sample.get("query_id", idx),
        "Claim": sample.get("claim", ""),
        "label": best_verdict,
        "Verdict_BoN": best_verdict,
        "BoN_Verdict_list": verdict_list,
        "Reasoning_traces": justification_list,
        "score_list": verifier_score_list,
    })

# Atomic write
tmp_path = PREDICTIONS_JSON + ".tmp"
with open(tmp_path, "w", encoding="utf-8") as fp:
    json.dump(predictions, fp, indent=4, ensure_ascii=False)
os.replace(tmp_path, PREDICTIONS_JSON)
print(f"Saved {len(predictions)} claim-level predictions to {PREDICTIONS_JSON}")

Running predictions on 2558 samples from ../English/clef_2026_final_english_test.json


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2438.40it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_62105/2767

Saved 2558 claim-level predictions to ./output_models/clef_predictions.json
